# Presentation 03 — Detection results on long audio

Figures summarising **what the model found when run on real multi-hour field recordings**: detections per file, detections per species, confidence distributions, and a time-of-day pattern.

Run cells in order. All generated figures are saved to `outputs/presentation/` on your Drive so you can drop them straight into slides.

## Setup — mount Drive, clone latest code, configure paths

This block is the same across all presentation notebooks. It mounts your Drive, pulls the latest branch, adds `src/` to `sys.path` and sets `PRIMATE_DATA_ROOT`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf primates-sound-detection
!git clone -q -b claude/review-repo-QAc6j https://github.com/Mo119m/primates-sound-detection.git
%cd /content/primates-sound-detection
!pip install -q -r requirements.txt

In [ ]:
import os, sys, importlib
PROJECT_ROOT = '/content/primates-sound-detection'
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
os.environ['PRIMATE_DATA_ROOT'] = '/content/drive/MyDrive/primates-data'

import config
importlib.reload(config)

PRESENT_DIR = os.path.join(config.OUTPUT_ROOT, 'presentation')
os.makedirs(PRESENT_DIR, exist_ok=True)
print('Presentation figures will be saved to:', PRESENT_DIR)

In [ ]:
# Consistent matplotlib style for every figure in this notebook
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 220,
    'savefig.bbox': 'tight',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'legend.frameon': False,
})

# Shared species color palette
SPECIES_COLORS = {
    'Cercopithecus_nictitans': '#E67E22',  # orange
    'Colobus_guereza':         '#27AE60',  # green
    'Pan_troglodytes':         '#2980B9',  # blue
    'Background':              '#7F8C8D',  # grey
}

def save_fig(fig, name):
    path = os.path.join(PRESENT_DIR, name)
    fig.savefig(path)
    print(f'  saved -> {path}')
    return path

## Step 1 — Load all detection CSVs

This reads every `*_detections.csv` under `outputs/detections/` produced by the main pipeline. **Run the main pipeline (Step 8b) first** to populate this folder at your chosen threshold.

In [ ]:
import glob
import pandas as pd
import re

det_dir = config.DETECTION_OUTPUT_DIR
csv_files = sorted(glob.glob(os.path.join(det_dir, '*_detections.csv')))
print(f'Found {len(csv_files)} detection CSVs in {det_dir}')

per_file = {}
for p in csv_files:
    fname = os.path.basename(p).replace('_detections.csv', '') + '.wav'
    df = pd.read_csv(p)
    per_file[fname] = df

# Concatenate into one big frame for aggregate plots
all_rows = []
for fname, df in per_file.items():
    if len(df) == 0:
        continue
    tmp = df.copy()
    tmp['source'] = fname
    all_rows.append(tmp)

all_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame(
    columns=['start_time', 'end_time', 'species', 'confidence', 'source']
)
print(f'Total detections across all files: {len(all_df)}')
if len(all_df) > 0:
    print(all_df['species'].value_counts().to_string())

## Figure 1 — Detections per long-audio file

Stacked bar: one bar per recording, colored by species.

In [ ]:
if len(all_df) == 0:
    print('No detections to plot.')
else:
    pivot = (all_df.groupby(['source', 'species']).size()
             .unstack(fill_value=0)
             .reindex(sorted(per_file.keys())))

    fig, ax = plt.subplots(figsize=(max(8, len(pivot) * 0.55), 5.5))
    bottom = np.zeros(len(pivot))
    for sp in pivot.columns:
        vals = pivot[sp].values
        ax.bar(range(len(pivot)), vals, bottom=bottom,
               color=SPECIES_COLORS.get(sp, '#888'), label=sp, edgecolor='black', linewidth=0.4)
        bottom = bottom + vals

    ax.set_xticks(range(len(pivot)))
    ax.set_xticklabels([s.replace('.wav', '') for s in pivot.index], rotation=40, ha='right', fontsize=9)
    ax.set_ylabel('Detections (after NMS)')
    ax.set_title('Detections per recording')
    ax.legend(loc='upper right')
    plt.tight_layout()
    save_fig(fig, '01_detections_per_file.png')
    plt.show()

## Figure 2 — Total detections per species

In [ ]:
if len(all_df) > 0:
    counts = all_df['species'].value_counts()
    colors = [SPECIES_COLORS.get(c, '#888') for c in counts.index]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='black', linewidth=0.8)
    ax.set_ylabel('Total detections')
    ax.set_title(f'Total detections across {len(per_file)} recordings')
    for b, v in zip(bars, counts.values):
        ax.text(b.get_x() + b.get_width()/2, v, str(int(v)),
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.margins(y=0.15)
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    save_fig(fig, '02_detections_per_species.png')
    plt.show()

## Figure 3 — Confidence distribution

Histogram of detection confidences for each species. Useful for discussing whether the threshold is set correctly.

In [ ]:
if len(all_df) > 0:
    species_list = sorted(all_df['species'].unique())
    fig, ax = plt.subplots(figsize=(10, 5.5))
    bins = np.linspace(all_df['confidence'].min(), 1.0, 25)
    for sp in species_list:
        sub = all_df[all_df['species'] == sp]['confidence']
        ax.hist(sub, bins=bins, alpha=0.65,
                label=f'{sp} (n={len(sub)})',
                color=SPECIES_COLORS.get(sp, '#888'), edgecolor='black', linewidth=0.4)
    ax.set_xlabel('Model confidence')
    ax.set_ylabel('Count')
    ax.set_title('Detection confidence distribution')
    ax.legend()
    plt.tight_layout()
    save_fig(fig, '03_confidence_histogram.png')
    plt.show()

## Figure 4 — Detection timeline per file

One row per recording; each detection is a colored dot placed at its start time (in seconds into the recording). Gives an at-a-glance picture of when in each file vocalisations happen.

In [ ]:
if len(all_df) > 0:
    files_sorted = sorted(per_file.keys())
    fig, ax = plt.subplots(figsize=(12, max(3, 0.45 * len(files_sorted))))
    for y, fname in enumerate(files_sorted):
        df = per_file[fname]
        if len(df) == 0:
            continue
        for sp in df['species'].unique():
            sub = df[df['species'] == sp]
            ax.scatter(sub['start_time'], [y] * len(sub),
                       s=40 + 120 * (sub['confidence'] - 0.3).clip(lower=0),
                       color=SPECIES_COLORS.get(sp, '#888'),
                       alpha=0.75, edgecolor='black', linewidth=0.3,
                       label=sp)
    # dedupe legend
    handles, labels = ax.get_legend_handles_labels()
    uniq = dict(zip(labels, handles))
    ax.legend(uniq.values(), uniq.keys(), loc='upper right')
    ax.set_yticks(range(len(files_sorted)))
    ax.set_yticklabels([f.replace('.wav', '') for f in files_sorted], fontsize=8)
    ax.set_xlabel('Time in recording (s)')
    ax.set_title('Detection timeline (dot size ∝ confidence)')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    save_fig(fig, '04_detection_timeline.png')
    plt.show()

## Figure 5 — Detections by time of day

The long-audio filenames encode a timestamp (`20220609T080000+0100_...`). We parse the hour out of each filename and aggregate detections per clock hour — useful for discussing diurnal vocal activity patterns.

If the filenames in your data don't match this pattern the cell will just print a warning and skip.

In [ ]:
if len(all_df) == 0:
    print('No detections to aggregate.')
else:
    ts_re = re.compile(r'(\d{8})T(\d{6})')
    per_hour = {}  # species -> {hour: count}
    n_parsed = 0
    for fname, df in per_file.items():
        m = ts_re.search(fname)
        if not m:
            continue
        hour = int(m.group(2)[:2])
        n_parsed += 1
        for sp in df['species'].unique() if len(df) > 0 else []:
            sub = df[df['species'] == sp]
            per_hour.setdefault(sp, {}).setdefault(hour, 0)
            per_hour[sp][hour] += len(sub)

    if n_parsed == 0:
        print('Could not parse any timestamps from filenames — skip this figure.')
    else:
        hours = list(range(24))
        fig, ax = plt.subplots(figsize=(11, 5))
        bottom = np.zeros(24)
        for sp, counts_by_hour in per_hour.items():
            vals = np.array([counts_by_hour.get(h, 0) for h in hours], dtype=float)
            ax.bar(hours, vals, bottom=bottom,
                   color=SPECIES_COLORS.get(sp, '#888'),
                   label=sp, edgecolor='black', linewidth=0.4)
            bottom = bottom + vals
        ax.set_xticks(hours)
        ax.set_xlabel('Hour of day (local time, from filename)')
        ax.set_ylabel('Detections')
        ax.set_title(f'Detections by hour of day  ({n_parsed} recordings parsed)')
        ax.legend()
        plt.tight_layout()
        save_fig(fig, '05_detections_by_hour.png')
        plt.show()

## Done

Figures `01_*.png` through `05_*.png` are on Drive under `outputs/presentation/`. If you re-run the main pipeline at a different threshold, just re-run this notebook from Step 1 and it'll pick up the new CSVs.